# Análisis Multi-Especie de Ocurrencias de Biodiversidad en GBIF (España y Portugal)

Este Jupyter Notebook automatiza la consulta, limpieza, enriquecimiento, exportación tabular, visualización geoespacial y **persistencia relacional (SQLite y MySQL 8)** de registros de presencia de biodiversidad desde la **API pública de GBIF** (estándar **Darwin Core**) para una **lista configurable de especies** en **España (`ES`)** y **Portugal (`PT`)**.

### Flujo de trabajo del pipeline:
1. **Resolución taxonómica dinámica (`/v1/species/match`)**: Convierte cada nombre científico de la lista en su identificador único `usageKey` (`taxonKey`) y verifica su rango taxonómico y nivel de coincidencia.
2. **Extracción paginada (`/v1/occurrence/search`)**: Descarga registros georreferenciados (`hasCoordinate=True`) por especie y país, controlando el límite máximo de registros por especie para evitar bloqueos en taxones muy abundantes.
3. **Normalización Darwin Core y limpieza de URLs**: Extrae los campos clave, construye el enlace directo a la ficha de GBIF (`https://www.gbif.org/occurrence/{gbifID}`), formatea correctamente múltiples URLs multimedia sin generar cadenas malformadas ni valores `NaN` inconsistentes y filtra coordenadas inválidas.
4. **Exportación individual y consolidada**: Guarda un archivo `.csv` independiente por especie y un archivo maestro consolidado (`gbif_multiespecie_es_pt.csv`), además de generar un resumen comparativo.
5. **Cartografía interactiva sin API Key (`folium`)**: Genera mapas HTML individuales con `MarkerCluster` y un mapa comparativo multicapa (`LayerControl`) usando teselas abiertas (OpenStreetMap y Esri World Imagery).
6. **Persistencia en Base de Datos Relacional (SQLite + MySQL 8)**: Crea automáticamente el esquema normalizado en 3FN (`taxones`, `ocurrencias`, `ocurrencia_media` y la vista `vw_ocurrencias_completas`) y realiza inserciones idempotentes (`UPSERT`) mediante consultas parametrizadas seguras frente a inyección SQL.

In [1]:
import re
import time
from pathlib import Path
from typing import Any

import folium
from folium.plugins import MarkerCluster
import pandas as pd
import truststore

# Netskope, en la red de la Junta, sustituye el certificado de GBIF.
# El almacén de Windows incluye esa CA; la verificación TLS sigue activa.
truststore.inject_into_ssl()
import requests

# ==============================================================================
# PARÁMETROS DE CONFIGURACIÓN DEL PIPELINE
# ==============================================================================
ESPECIES_OBJETIVO = [
    "Canis lupus",
    "Lynx pardinus",
    "Ursus arctos",
    "Gypaetus barbatus",
]

PAISES = ["ES", "PT"]
MAX_REGISTROS_POR_ESPECIE_PAIS = 600  # Límite por país para evitar descargas masivas
TAMANO_PAGINA = 300                   # Máximo permitido por petición en GBIF (300)
CARPETA_SALIDA = (Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent) / "salidas_gbif_multiespecie"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

HEADERS_HTTP = {
    "User-Agent": "GBIF-MultiSpecies-Biodiversity-Pipeline/1.0 (Python requests)"
}

COLUMNAS_DARWIN_CORE = [
    "especie_buscada",
    "gbifID",
    "url_ocurrencia_gbif",
    "scientificName",
    "acceptedScientificName",
    "taxonRank",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
    "countryCode",
    "stateProvince",
    "locality",
    "decimalLatitude",
    "decimalLongitude",
    "coordinateUncertaintyInMeters",
    "eventDate",
    "year",
    "month",
    "day",
    "basisOfRecord",
    "institutionCode",
    "collectionCode",
    "datasetKey",
    "occurrenceID",
    "recordedBy",
    "informationWithheld",
    "media_urls",
]

print(f"Configuración cargada. Especies a procesar: {len(ESPECIES_OBJETIVO)}")
print(f"Directorio de salida: {CARPETA_SALIDA.resolve()}")

Configuración cargada. Especies a procesar: 4
Directorio de salida: E:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie


## 1. Resolución Taxonómica Dinámica (`/v1/species/match`)

Antes de consultar ocurrencias, cada nombre científico se valida contra el *Backbone Taxonomy* de GBIF. Esto garantiza obtener el `usageKey` exacto y detectar si el nombre corresponde a una especie, subespecie o sinonimia.

In [2]:
def normalizar_slug(nombre: str) -> str:
    """Convierte un nombre científico en un identificador seguro para nombres de archivo."""
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", nombre.strip().lower()).strip("_")
    return slug


def resolver_taxon_gbif(nombre_cientifico: str) -> dict[str, Any]:
    """Consulta /v1/species/match para obtener el usageKey y metadatos taxonómicos."""
    url = "https://api.gbif.org/v1/species/match"
    params = {"name": nombre_cientifico, "strict": False}
    
    resp = requests.get(url, params=params, headers=HEADERS_HTTP, timeout=20)
    resp.raise_for_status()
    data = resp.json()
    
    return {
        "especie_buscada": nombre_cientifico,
        "slug": normalizar_slug(nombre_cientifico),
        "usageKey": data.get("usageKey"),
        "scientificName": data.get("scientificName"),
        "canonicalName": data.get("canonicalName"),
        "rank": data.get("rank"),
        "status": data.get("status"),
        "matchType": data.get("matchType"),
        "confidence": data.get("confidence"),
    }


catalogo_taxones = []
for especie in ESPECIES_OBJETIVO:
    info = resolver_taxon_gbif(especie)
    catalogo_taxones.append(info)

df_catalogo = pd.DataFrame(catalogo_taxones)
display(df_catalogo)

,especie_buscada,slug,usageKey,scientificName,canonicalName,rank,status,matchType,confidence
0,Canis lupus,canis_lupus,5219173,"Canis lupus Linnaeus, 1758",Canis lupus,SPECIES,ACCEPTED,EXACT,99
1,Lynx pardinus,lynx_pardinus,2435261,"Lynx pardinus (Temminck, 1827)",Lynx pardinus,SPECIES,ACCEPTED,EXACT,99
2,Ursus arctos,ursus_arctos,2433433,"Ursus arctos Linnaeus, 1758",Ursus arctos,SPECIES,ACCEPTED,EXACT,99
3,Gypaetus barbatus,gypaetus_barbatus,2480649,"Gypaetus barbatus (Linnaeus, 1758)",Gypaetus barbatus,SPECIES,ACCEPTED,EXACT,99


## 2. Extracción Paginada y Limpieza bajo Estándar Darwin Core

Para cada especie resuelta con `usageKey` válido:
* Se consulta `/v1/occurrence/search` filtrando por `taxonKey`, `country` (`ES` y `PT`) y `hasCoordinate=True`.
* Se extraen de forma segura los enlaces multimedia del array `media` (evitando comas sueltas, corchetes o cadenas malformadas al exportar a CSV).
* Se añade la columna `url_ocurrencia_gbif` apuntando directamente a `https://www.gbif.org/occurrence/{gbifID}`.

In [3]:
def extraer_urls_multimedia(registro: dict[str, Any]) -> str:
    """Extrae y limpia las URLs multimedia de un registro GBIF separándolas por ' | '."""
    media_lista = registro.get("media")
    if not isinstance(media_lista, list) or not media_lista:
        return ""
    
    urls_limpias = []
    for item in media_lista:
        if isinstance(item, dict):
            url = item.get("identifier") or item.get("references")
            if isinstance(url, str) and url.strip().startswith(("http://", "https://")):
                url_norm = url.strip()
                if url_norm not in urls_limpias:
                    urls_limpias.append(url_norm)
    return " | ".join(urls_limpias)


def descargar_ocurrencias_taxon(
    taxon_info: dict[str, Any],
    paises: list[str] = PAISES,
    max_por_pais: int = MAX_REGISTROS_POR_ESPECIE_PAIS,
) -> pd.DataFrame:
    """Descarga ocurrencias georreferenciadas de GBIF para un taxón y lista de países."""
    usage_key = taxon_info.get("usageKey")
    especie_buscada = taxon_info["especie_buscada"]
    
    if pd.isna(usage_key) or usage_key is None:
        print(f"[AVISO] No se encontró usageKey válido para '{especie_buscada}'. Se omite.")
        return pd.DataFrame(columns=COLUMNAS_DARWIN_CORE)
    
    url_search = "https://api.gbif.org/v1/occurrence/search"
    registros_procesados = []
    
    for pais in paises:
        offset = 0
        descargados_pais = 0
        
        while descargados_pais < max_por_pais:
            limit = min(TAMANO_PAGINA, max_por_pais - descargados_pais)
            params = {
                "taxonKey": int(usage_key),
                "country": pais,
                "hasCoordinate": "true",
                "hasGeospatialIssue": "false",
                "limit": limit,
                "offset": offset,
            }
            
            resp = requests.get(url_search, params=params, headers=HEADERS_HTTP, timeout=25)
            resp.raise_for_status()
            payload = resp.json()
            resultados = payload.get("results", [])
            
            if not resultados:
                break
            
            for reg in resultados:
                gbif_id = reg.get("gbifID")
                fila = {
                    "especie_buscada": especie_buscada,
                    "usage_key": int(usage_key),
                    "gbifID": gbif_id,
                    "url_ocurrencia_gbif": f"https://www.gbif.org/occurrence/{gbif_id}" if gbif_id else "",
                    "media_urls": extraer_urls_multimedia(reg),
                }
                for col in COLUMNAS_DARWIN_CORE:
                    if col not in fila:
                        fila[col] = reg.get(col)
                registros_procesados.append(fila)
            
            descargados_pais += len(resultados)
            offset += len(resultados)
            
            if payload.get("endOfRecords", True):
                break
            time.sleep(0.15)
            
        print(f"  - {especie_buscada} [{pais}]: {descargados_pais} registros descargados.")
    
    if not registros_procesados:
        return pd.DataFrame(columns=COLUMNAS_DARWIN_CORE)
    
    cols_con_fk = ["usage_key"] + COLUMNAS_DARWIN_CORE
    df = pd.DataFrame(registros_procesados)[cols_con_fk]
    
    # Limpieza de tipos numéricos y coordenadas válidas
    df["decimalLatitude"] = pd.to_numeric(df["decimalLatitude"], errors="coerce")
    df["decimalLongitude"] = pd.to_numeric(df["decimalLongitude"], errors="coerce")
    df["coordinateUncertaintyInMeters"] = pd.to_numeric(
        df["coordinateUncertaintyInMeters"], errors="coerce"
    )
    df = df.dropna(subset=["decimalLatitude", "decimalLongitude"]).copy()
    df = df.drop_duplicates(subset=["gbifID"]).reset_index(drop=True)
    
    return df

## 3. Ejecución en Lote, Exportación de CSVs y Resumen Comparativo

Se recorre el catálogo taxonómico, se descarga y limpia cada especie, se exporta su CSV individual (`gbif_<especie>_es_pt.csv`) y se consolida un DataFrame maestro (`df_total`) para análisis comparativo.

In [4]:
dataframes_por_especie: dict[str, pd.DataFrame] = {}
lista_dfs_consolidados: list[pd.DataFrame] = []

for taxon in catalogo_taxones:
    especie = taxon["especie_buscada"]
    slug = taxon["slug"]
    print(f"Procesando: {especie} (usageKey={taxon['usageKey']})...")
    
    df_esp = descargar_ocurrencias_taxon(taxon)
    dataframes_por_especie[especie] = df_esp
    
    if not df_esp.empty:
        ruta_csv = CARPETA_SALIDA / f"gbif_{slug}_es_pt.csv"
        df_esp.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
        lista_dfs_consolidados.append(df_esp)
        print(f"  -> Guardado: {ruta_csv} ({len(df_esp)} filas válidas)\n")
    else:
        print(f"  -> Sin registros georreferenciados para {especie}.\n")

if lista_dfs_consolidados:
    df_total = pd.concat(lista_dfs_consolidados, ignore_index=True)
    ruta_csv_total = CARPETA_SALIDA / "gbif_multiespecie_es_pt.csv"
    df_total.to_csv(ruta_csv_total, index=False, encoding="utf-8-sig")
    print(f"Archivo maestro consolidado guardado en: {ruta_csv_total} ({len(df_total)} registros totales)")
else:
    df_total = pd.DataFrame(columns=["usage_key"] + COLUMNAS_DARWIN_CORE)
    print("No se obtuvieron registros para ninguna especie de la lista.")

# Tabla resumen comparativa por especie y país
if not df_total.empty:
    resumen = (
        df_total.groupby(["especie_buscada", "countryCode"], dropna=False)
        .agg(
            total_registros=("gbifID", "count"),
            con_multimedia=("media_urls", lambda s: (s.fillna("") != "").sum()),
            incertidumbre_mediana_m=("coordinateUncertaintyInMeters", "median"),
            anio_min=("year", "min"),
            anio_max=("year", "max"),
        )
        .reset_index()
    )
    display(resumen)

Procesando: Canis lupus (usageKey=5219173)...
  - Canis lupus [ES]: 600 registros descargados.
  - Canis lupus [PT]: 600 registros descargados.
  -> Guardado: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\gbif_canis_lupus_es_pt.csv (1200 filas válidas)

Procesando: Lynx pardinus (usageKey=2435261)...
  - Lynx pardinus [ES]: 600 registros descargados.
  - Lynx pardinus [PT]: 600 registros descargados.
  -> Guardado: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\gbif_lynx_pardinus_es_pt.csv (1200 filas válidas)

Procesando: Ursus arctos (usageKey=2433433)...
  - Ursus arctos [ES]: 600 registros descargados.
  - Ursus arctos [PT]: 0 registros descargados.
  -> Guardado: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\gbif_ursus_arctos_es_pt.csv (600 filas válidas)

Procesando: Gypaetus barbatus (usageKey=2480649)...
  - Gypaetus barbatus [ES]: 600 registros descargados.
  - Gypaetus b

,especie_buscada,countryCode,total_registros,con_multimedia,incertidumbre_mediana_m,anio_min,anio_max
0,Canis lupus,ES,600,447,50.0,2025.0,2026.0
1,Canis lupus,PT,600,520,61.0,2015.0,2026.0
2,Gypaetus barbatus,ES,600,250,12.0,2025.0,2026.0
3,Gypaetus barbatus,PT,57,0,NaN,NaN,NaN
4,Lynx pardinus,ES,600,332,28270.0,2010.0,2026.0
5,Lynx pardinus,PT,600,32,530.0,1944.0,2026.0
6,Ursus arctos,ES,600,320,27537.0,2007.0,2026.0


## 4. Cartografía Interactiva Multi-Especie (`Folium`)

Para evitar problemas con claves de API externas, todos los mapas utilizan teselas abiertas (**OpenStreetMap** y **Esri World Imagery**). Se generan:
1. **Un mapa HTML individual por especie** con agrupación dinámica (`MarkerCluster`) y popups enriquecidos (taxón, fecha, base del registro, incertidumbre espacial y enlace directo a GBIF).
2. **Un mapa comparativo multi-especie (`mapa_comparativo_multiespecie_es_pt.html`)** donde cada especie cuenta con su propia capa activable/desactivable (`folium.FeatureGroup` + `folium.LayerControl`) y un color distintivo.

In [5]:
PALETA_COLORES = [
    "darkred",
    "darkblue",
    "darkgreen",
    "purple",
    "orange",
    "cadetblue",
    "darkpurple",
    "pink",
]


def crear_popup_html(fila: pd.Series) -> folium.Popup:
    """Construye un popup HTML limpio con enlace directo a la ocurrencia en GBIF."""
    fecha = fila.get("eventDate") if pd.notna(fila.get("eventDate")) else "Sin fecha"
    incertidumbre = (
        f"{int(fila['coordinateUncertaintyInMeters'])} m"
        if pd.notna(fila.get("coordinateUncertaintyInMeters"))
        else "No especificada"
    )
    basis = fila.get("basisOfRecord") if pd.notna(fila.get("basisOfRecord")) else "N/D"
    url_gbif = fila.get("url_ocurrencia_gbif", "")
    
    media_html = ""
    media_str = str(fila.get("media_urls") or "").strip()
    if media_str:
        primera_img = media_str.split(" | ")[0]
        media_html = f'<br><a href="{primera_img}" target="_blank">Ver evidencia multimedia</a>'

    html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; min-width: 220px;">
        <b>{fila.get('scientificName', fila.get('especie_buscada'))}</b><br>
        <b>País:</b> {fila.get('countryCode', 'N/D')}<br>
        <b>Fecha:</b> {fecha}<br>
        <b>Tipo registro:</b> {basis}<br>
        <b>Incertidumbre:</b> {incertidumbre}<br>
        <a href="{url_gbif}" target="_blank">Abrir ficha en GBIF ({fila.get('gbifID')})</a>
        {media_html}
    </div>
    """
    return folium.Popup(html, max_width=300)


def agregar_capas_base(mapa: folium.Map) -> None:
    """Añade capas base abiertas sin requerir API key."""
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satélite (Esri)",
        overlay=False,
        control=True,
    ).add_to(mapa)


# 4.1 Mapas individuales por especie
for idx, (especie, df_esp) in enumerate(dataframes_por_especie.items()):
    if df_esp.empty:
        continue
    
    slug = normalizar_slug(especie)
    color = PALETA_COLORES[idx % len(PALETA_COLORES)]
    centro_lat = df_esp["decimalLatitude"].mean()
    centro_lon = df_esp["decimalLongitude"].mean()
    
    mapa_ind = folium.Map(
        location=[centro_lat, centro_lon],
        zoom_start=6,
        tiles="OpenStreetMap",
    )
    agregar_capas_base(mapa_ind)
    
    cluster = MarkerCluster(name=f"{especie} ({len(df_esp)})").add_to(mapa_ind)
    for _, fila in df_esp.iterrows():
        folium.Marker(
            location=[fila["decimalLatitude"], fila["decimalLongitude"]],
            popup=crear_popup_html(fila),
            tooltip=f"{especie} ({fila['countryCode']})",
            icon=folium.Icon(color=color, icon="paw", prefix="fa"),
        ).add_to(cluster)
    
    folium.LayerControl(collapsed=False).add_to(mapa_ind)
    ruta_mapa = CARPETA_SALIDA / f"mapa_{slug}_es_pt.html"
    mapa_ind.save(str(ruta_mapa))
    print(f"Mapa individual guardado: {ruta_mapa}")


# 4.2 Mapa comparativo unificado con una capa por especie
if not df_total.empty:
    mapa_comparativo = folium.Map(
        location=[40.2, -4.5],  # Centro aproximado Península Ibérica
        zoom_start=6,
        tiles="OpenStreetMap",
    )
    agregar_capas_base(mapa_comparativo)
    
    for idx, (especie, df_esp) in enumerate(dataframes_por_especie.items()):
        if df_esp.empty:
            continue
        color = PALETA_COLORES[idx % len(PALETA_COLORES)]
        capa_especie = folium.FeatureGroup(name=f"{especie} ({len(df_esp)})", show=True)
        cluster_esp = MarkerCluster().add_to(capa_especie)
        
        for _, fila in df_esp.iterrows():
            folium.CircleMarker(
                location=[fila["decimalLatitude"], fila["decimalLongitude"]],
                radius=6,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.75,
                popup=crear_popup_html(fila),
                tooltip=f"{especie} | {fila.get('eventDate', 's/f')}",
            ).add_to(cluster_esp)
            
        capa_especie.add_to(mapa_comparativo)
    
    folium.LayerControl(collapsed=False).add_to(mapa_comparativo)
    ruta_mapa_comp = CARPETA_SALIDA / "mapa_comparativo_multiespecie_es_pt.html"
    mapa_comparativo.save(str(ruta_mapa_comp))
    print(f"\nMapa comparativo multi-especie guardado: {ruta_mapa_comp}")
    display(mapa_comparativo)

Mapa individual guardado: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\mapa_canis_lupus_es_pt.html
Mapa individual guardado: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\mapa_lynx_pardinus_es_pt.html
Mapa individual guardado: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\mapa_ursus_arctos_es_pt.html
Mapa individual guardado: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\mapa_gypaetus_barbatus_es_pt.html

Mapa comparativo multi-especie guardado: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\mapa_comparativo_multiespecie_es_pt.html


## 5. Persistencia Relacional en SQLite y MySQL 8 (Modelo Normalizado 3FN)

Esta sección crea la estructura relacional (`taxones`, `ocurrencias`, `ocurrencia_media` y la vista analítica `vw_ocurrencias_completas`) y carga los datos descargados de forma idempotente (*upsert*) utilizando exclusivamente **consultas parametrizadas** para prevenir inyección SQL.

* **SQLite**: Se genera automáticamente en `salidas_gbif_multiespecie/biodiversidad_gbif.db` sin requerir configuración adicional.
* **MySQL 8**: Lee las credenciales desde un archivo `.env` local (`MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, `MYSQL_DATABASE`). Si `ACTIVAR_MYSQL = True` y el servidor está accesible, sincroniza también la base de datos en MySQL.

In [ ]:
import os
import sqlite3
from dotenv import load_dotenv

try:
    import mysql.connector
    from mysql.connector import Error as MySQLError
    MYSQL_DISPONIBLE = True
except ImportError:
    MYSQL_DISPONIBLE = False
    print("[INFO] 'mysql-connector-python' no está instalado. Se ejecutará solo SQLite (pip install mysql-connector-python python-dotenv para habilitar MySQL).")

load_dotenv()

# Configuración de destino de Base de Datos
ACTIVAR_SQLITE = True
RUTA_SQLITE = CARPETA_SALIDA / "biodiversidad_gbif.db"

ACTIVAR_MYSQL = True  # Cambiar a False si solo se desea usar SQLite
MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "127.0.0.1"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD", ""),
    "database": os.getenv("MYSQL_DATABASE", "biodiversidad_gbif"),
}


def limpiar_valor_sql(valor: Any) -> Any:
    """Convierte NaN/NaT de pandas a None nativo de Python para SQL."""
    if pd.isna(valor):
        return None
    if hasattr(valor, "item"):
        return valor.item()
    return valor


def preparar_registros_para_bd(
    df_cat: pd.DataFrame, df_oc: pd.DataFrame
) -> tuple[list[tuple], list[tuple], list[tuple]]:
    """Prepara tuplas limpias para taxones, ocurrencias y multimedia."""
    # Mapear jerarquía taxonómica desde df_oc hacia df_cat
    jerarquia_cols = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
    jerarquia_por_key = {}
    if not df_oc.empty:
        for u_key, grupo in df_oc.groupby("usage_key"):
            primera = grupo.iloc[0]
            jerarquia_por_key[int(u_key)] = {c: limpiar_valor_sql(primera.get(c)) for c in jerarquia_cols}

    tuplas_taxones = []
    for _, row in df_cat.iterrows():
        u_key = limpiar_valor_sql(row.get("usageKey"))
        if u_key is None:
            continue
        u_key = int(u_key)
        jer = jerarquia_por_key.get(u_key, {})
        tuplas_taxones.append((
            u_key,
            limpiar_valor_sql(row.get("especie_buscada")),
            limpiar_valor_sql(row.get("scientificName")),
            limpiar_valor_sql(row.get("canonicalName")),
            limpiar_valor_sql(row.get("rank")),
            limpiar_valor_sql(row.get("status")),
            jer.get("kingdom"),
            jer.get("phylum"),
            jer.get("class"),
            jer.get("order"),
            jer.get("family"),
            jer.get("genus"),
            jer.get("species"),
        ))

    tuplas_ocurrencias = []
    tuplas_media = []
    for _, row in df_oc.iterrows():
        gbif_id = int(row["gbifID"])
        tuplas_ocurrencias.append((
            gbif_id,
            int(row["usage_key"]),
            limpiar_valor_sql(row.get("url_ocurrencia_gbif")),
            limpiar_valor_sql(row.get("scientificName")),
            limpiar_valor_sql(row.get("acceptedScientificName")),
            limpiar_valor_sql(row.get("taxonRank")),
            limpiar_valor_sql(row.get("countryCode")),
            limpiar_valor_sql(row.get("stateProvince")),
            limpiar_valor_sql(row.get("locality")),
            float(row["decimalLatitude"]),
            float(row["decimalLongitude"]),
            limpiar_valor_sql(row.get("coordinateUncertaintyInMeters")),
            limpiar_valor_sql(row.get("eventDate")),
            limpiar_valor_sql(row.get("year")),
            limpiar_valor_sql(row.get("month")),
            limpiar_valor_sql(row.get("day")),
            limpiar_valor_sql(row.get("basisOfRecord")),
            limpiar_valor_sql(row.get("institutionCode")),
            limpiar_valor_sql(row.get("collectionCode")),
            limpiar_valor_sql(row.get("datasetKey")),
            limpiar_valor_sql(row.get("occurrenceID")),
            limpiar_valor_sql(row.get("recordedBy")),
            limpiar_valor_sql(row.get("informationWithheld")),
        ))

        media_str = str(row.get("media_urls") or "").strip()
        if media_str:
            for idx_m, url_m in enumerate(media_str.split(" | "), start=1):
                if url_m.strip():
                    tuplas_media.append((gbif_id, url_m.strip(), idx_m))

    return tuplas_taxones, tuplas_ocurrencias, tuplas_media


def cargar_en_sqlite(
    ruta_db: Path,
    tuplas_taxones: list[tuple],
    tuplas_ocurrencias: list[tuple],
    tuplas_media: list[tuple],
) -> None:
    """Crea el esquema en SQLite e inserta/actualiza los registros de forma parametrizada."""
    ddl_sqlite = """
    PRAGMA foreign_keys = ON;

    CREATE TABLE IF NOT EXISTS taxones (
        usage_key INTEGER PRIMARY KEY,
        especie_buscada TEXT NOT NULL,
        scientific_name TEXT NOT NULL,
        canonical_name TEXT,
        taxon_rank TEXT,
        taxonomic_status TEXT,
        kingdom TEXT,
        phylum TEXT,
        class_name TEXT,
        order_name TEXT,
        family TEXT,
        genus TEXT,
        species TEXT,
        fecha_creacion TEXT NOT NULL DEFAULT (datetime('now'))
    );

    CREATE INDEX IF NOT EXISTS idx_taxones_especie_buscada ON taxones (especie_buscada);

    CREATE TABLE IF NOT EXISTS ocurrencias (
        gbif_id INTEGER PRIMARY KEY,
        usage_key INTEGER NOT NULL,
        url_ocurrencia_gbif TEXT NOT NULL,
        scientific_name_registrado TEXT,
        accepted_scientific_name TEXT,
        taxon_rank TEXT,
        country_code TEXT NOT NULL CHECK (length(country_code) = 2),
        state_province TEXT,
        locality TEXT,
        decimal_latitude REAL NOT NULL,
        decimal_longitude REAL NOT NULL,
        coordinate_uncertainty_m REAL,
        event_date TEXT,
        event_year INTEGER,
        event_month INTEGER,
        event_day INTEGER,
        basis_of_record TEXT,
        institution_code TEXT,
        collection_code TEXT,
        dataset_key TEXT,
        occurrence_id TEXT,
        recorded_by TEXT,
        information_withheld TEXT,
        fecha_ingesta TEXT NOT NULL DEFAULT (datetime('now')),
        FOREIGN KEY (usage_key) REFERENCES taxones (usage_key)
            ON UPDATE CASCADE ON DELETE RESTRICT
    );

    CREATE INDEX IF NOT EXISTS idx_ocurrencias_taxon_pais ON ocurrencias (usage_key, country_code);
    CREATE INDEX IF NOT EXISTS idx_ocurrencias_coords ON ocurrencias (decimal_latitude, decimal_longitude);
    CREATE INDEX IF NOT EXISTS idx_ocurrencias_anio ON ocurrencias (event_year);

    CREATE TABLE IF NOT EXISTS ocurrencia_media (
        id_media INTEGER PRIMARY KEY AUTOINCREMENT,
        gbif_id INTEGER NOT NULL,
        media_url TEXT NOT NULL,
        orden INTEGER NOT NULL DEFAULT 1,
        UNIQUE (gbif_id, orden),
        FOREIGN KEY (gbif_id) REFERENCES ocurrencias (gbif_id)
            ON UPDATE CASCADE ON DELETE CASCADE
    );

    CREATE VIEW IF NOT EXISTS vw_ocurrencias_completas AS
    SELECT
        o.gbif_id,
        t.especie_buscada,
        t.canonical_name,
        o.scientific_name_registrado,
        o.country_code,
        o.state_province,
        o.decimal_latitude,
        o.decimal_longitude,
        o.coordinate_uncertainty_m,
        o.event_date,
        o.event_year,
        o.basis_of_record,
        o.url_ocurrencia_gbif,
        COUNT(m.id_media) AS total_imagenes,
        MIN(m.media_url) AS imagen_principal
    FROM ocurrencias o
    INNER JOIN taxones t ON o.usage_key = t.usage_key
    LEFT JOIN ocurrencia_media m ON o.gbif_id = m.gbif_id
    GROUP BY
        o.gbif_id, t.especie_buscada, t.canonical_name,
        o.scientific_name_registrado, o.country_code, o.state_province,
        o.decimal_latitude, o.decimal_longitude, o.coordinate_uncertainty_m,
        o.event_date, o.event_year, o.basis_of_record, o.url_ocurrencia_gbif;
    """

    sql_upsert_taxon = """
    INSERT INTO taxones (
        usage_key, especie_buscada, scientific_name, canonical_name, taxon_rank,
        taxonomic_status, kingdom, phylum, class_name, order_name, family, genus, species
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ON CONFLICT(usage_key) DO UPDATE SET
        especie_buscada = excluded.especie_buscada,
        scientific_name = excluded.scientific_name,
        canonical_name = excluded.canonical_name,
        taxon_rank = excluded.taxon_rank,
        taxonomic_status = excluded.taxonomic_status,
        kingdom = COALESCE(excluded.kingdom, taxones.kingdom),
        phylum = COALESCE(excluded.phylum, taxones.phylum),
        class_name = COALESCE(excluded.class_name, taxones.class_name),
        order_name = COALESCE(excluded.order_name, taxones.order_name),
        family = COALESCE(excluded.family, taxones.family),
        genus = COALESCE(excluded.genus, taxones.genus),
        species = COALESCE(excluded.species, taxones.species);
    """

    sql_upsert_ocurrencia = """
    INSERT INTO ocurrencias (
        gbif_id, usage_key, url_ocurrencia_gbif, scientific_name_registrado,
        accepted_scientific_name, taxon_rank, country_code, state_province, locality,
        decimal_latitude, decimal_longitude, coordinate_uncertainty_m, event_date,
        event_year, event_month, event_day, basis_of_record, institution_code,
        collection_code, dataset_key, occurrence_id, recorded_by, information_withheld
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ON CONFLICT(gbif_id) DO UPDATE SET
        usage_key = excluded.usage_key,
        scientific_name_registrado = excluded.scientific_name_registrado,
        accepted_scientific_name = excluded.accepted_scientific_name,
        state_province = excluded.state_province,
        locality = excluded.locality,
        decimal_latitude = excluded.decimal_latitude,
        decimal_longitude = excluded.decimal_longitude,
        coordinate_uncertainty_m = excluded.coordinate_uncertainty_m,
        event_date = excluded.event_date,
        basis_of_record = excluded.basis_of_record,
        fecha_ingesta = datetime('now');
    """

    sql_upsert_media = """
    INSERT INTO ocurrencia_media (gbif_id, media_url, orden)
    VALUES (?, ?, ?)
    ON CONFLICT(gbif_id, orden) DO UPDATE SET
        media_url = excluded.media_url;
    """

    with sqlite3.connect(ruta_db) as conn:
        conn.executescript(ddl_sqlite)
        conn.execute("PRAGMA foreign_keys = ON;")
        cursor = conn.cursor()
        cursor.executemany(sql_upsert_taxon, tuplas_taxones)
        cursor.executemany(sql_upsert_ocurrencia, tuplas_ocurrencias)
        cursor.executemany(sql_upsert_media, tuplas_media)
        conn.commit()

        resumen_db = pd.read_sql_query(
            "SELECT especie_buscada, country_code, COUNT(*) AS filas_en_bd, SUM(total_imagenes) AS imgs FROM vw_ocurrencias_completas GROUP BY especie_buscada, country_code;",
            conn,
        )
    print(f"[SQLite] Base de datos actualizada en: {ruta_db.resolve()}")
    display(resumen_db)


def cargar_en_mysql(
    config: dict[str, Any],
    tuplas_taxones: list[tuple],
    tuplas_ocurrencias: list[tuple],
    tuplas_media: list[tuple],
) -> None:
    """Crea la base de datos y tablas en MySQL 8 e inserta/actualiza registros de forma segura."""
    if not MYSQL_DISPONIBLE:
        return

    db_name = config["database"]
    if not re.match(r"^[a-zA-Z0-9_]+$", db_name):
        raise ValueError(f"Nombre de base de datos MySQL no válido: {db_name}")

    conn_servidor = None
    conn_db = None
    try:
        # 1. Conectar al servidor para asegurar que existe la base de datos
        cfg_sin_db = {k: v for k, v in config.items() if k != "database"}
        conn_servidor = mysql.connector.connect(**cfg_sin_db)
        cur_srv = conn_servidor.cursor()
        cur_srv.execute(
            f"CREATE DATABASE IF NOT EXISTS `{db_name}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"
        )
        cur_srv.close()
        conn_servidor.close()

        # 2. Conectar a la base de datos destino y crear tablas
        conn_db = mysql.connector.connect(**config)
        cursor = conn_db.cursor()

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS taxones (
            usage_key BIGINT UNSIGNED NOT NULL,
            especie_buscada VARCHAR(150) NOT NULL,
            scientific_name VARCHAR(255) NOT NULL,
            canonical_name VARCHAR(200) NULL,
            taxon_rank VARCHAR(50) NULL,
            taxonomic_status VARCHAR(50) NULL,
            kingdom VARCHAR(100) NULL,
            phylum VARCHAR(100) NULL,
            class_name VARCHAR(100) NULL,
            order_name VARCHAR(100) NULL,
            family VARCHAR(100) NULL,
            genus VARCHAR(100) NULL,
            species VARCHAR(150) NULL,
            fecha_creacion TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (usage_key),
            INDEX idx_taxones_especie_buscada (especie_buscada)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS ocurrencias (
            gbif_id BIGINT UNSIGNED NOT NULL,
            usage_key BIGINT UNSIGNED NOT NULL,
            url_ocurrencia_gbif VARCHAR(255) NOT NULL,
            scientific_name_registrado VARCHAR(255) NULL,
            accepted_scientific_name VARCHAR(255) NULL,
            taxon_rank VARCHAR(50) NULL,
            country_code CHAR(2) NOT NULL,
            state_province VARCHAR(150) NULL,
            locality TEXT NULL,
            decimal_latitude DECIMAL(10, 7) NOT NULL,
            decimal_longitude DECIMAL(10, 7) NOT NULL,
            coordinate_uncertainty_m DOUBLE NULL,
            event_date VARCHAR(50) NULL,
            event_year SMALLINT UNSIGNED NULL,
            event_month TINYINT UNSIGNED NULL,
            event_day TINYINT UNSIGNED NULL,
            basis_of_record VARCHAR(60) NULL,
            institution_code VARCHAR(100) NULL,
            collection_code VARCHAR(100) NULL,
            dataset_key VARCHAR(64) NULL,
            occurrence_id VARCHAR(255) NULL,
            recorded_by TEXT NULL,
            information_withheld TEXT NULL,
            fecha_ingesta TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (gbif_id),
            CONSTRAINT fk_ocurrencias_taxon
                FOREIGN KEY (usage_key) REFERENCES taxones (usage_key)
                ON UPDATE CASCADE ON DELETE RESTRICT,
            INDEX idx_ocurrencias_taxon_pais (usage_key, country_code),
            INDEX idx_ocurrencias_coords (decimal_latitude, decimal_longitude),
            INDEX idx_ocurrencias_anio (event_year)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS ocurrencia_media (
            id_media BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
            gbif_id BIGINT UNSIGNED NOT NULL,
            media_url VARCHAR(1024) NOT NULL,
            orden SMALLINT UNSIGNED NOT NULL DEFAULT 1,
            PRIMARY KEY (id_media),
            UNIQUE KEY uq_ocurrencia_media_orden (gbif_id, orden),
            CONSTRAINT fk_media_ocurrencia
                FOREIGN KEY (gbif_id) REFERENCES ocurrencias (gbif_id)
                ON UPDATE CASCADE ON DELETE CASCADE
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """)

        cursor.execute("""
        CREATE OR REPLACE VIEW vw_ocurrencias_completas AS
        SELECT
            o.gbif_id,
            t.especie_buscada,
            t.canonical_name,
            o.scientific_name_registrado,
            o.country_code,
            o.state_province,
            o.decimal_latitude,
            o.decimal_longitude,
            o.coordinate_uncertainty_m,
            o.event_date,
            o.event_year,
            o.basis_of_record,
            o.url_ocurrencia_gbif,
            COUNT(m.id_media) AS total_imagenes,
            MIN(m.media_url) AS imagen_principal
        FROM ocurrencias o
        INNER JOIN taxones t ON o.usage_key = t.usage_key
        LEFT JOIN ocurrencia_media m ON o.gbif_id = m.gbif_id
        GROUP BY
            o.gbif_id, t.especie_buscada, t.canonical_name,
            o.scientific_name_registrado, o.country_code, o.state_province,
            o.decimal_latitude, o.decimal_longitude, o.coordinate_uncertainty_m,
            o.event_date, o.event_year, o.basis_of_record, o.url_ocurrencia_gbif;
        """)

        sql_mysql_taxon = """
        INSERT INTO taxones (
            usage_key, especie_buscada, scientific_name, canonical_name, taxon_rank,
            taxonomic_status, kingdom, phylum, class_name, order_name, family, genus, species
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            especie_buscada = VALUES(especie_buscada),
            scientific_name = VALUES(scientific_name),
            canonical_name = VALUES(canonical_name),
            taxon_rank = VALUES(taxon_rank),
            taxonomic_status = VALUES(taxonomic_status),
            kingdom = COALESCE(VALUES(kingdom), kingdom),
            phylum = COALESCE(VALUES(phylum), phylum),
            class_name = COALESCE(VALUES(class_name), class_name),
            order_name = COALESCE(VALUES(order_name), order_name),
            family = COALESCE(VALUES(family), family),
            genus = COALESCE(VALUES(genus), genus),
            species = COALESCE(VALUES(species), species);
        """

        sql_mysql_ocurrencia = """
        INSERT INTO ocurrencias (
            gbif_id, usage_key, url_ocurrencia_gbif, scientific_name_registrado,
            accepted_scientific_name, taxon_rank, country_code, state_province, locality,
            decimal_latitude, decimal_longitude, coordinate_uncertainty_m, event_date,
            event_year, event_month, event_day, basis_of_record, institution_code,
            collection_code, dataset_key, occurrence_id, recorded_by, information_withheld
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            usage_key = VALUES(usage_key),
            scientific_name_registrado = VALUES(scientific_name_registrado),
            accepted_scientific_name = VALUES(accepted_scientific_name),
            state_province = VALUES(state_province),
            locality = VALUES(locality),
            decimal_latitude = VALUES(decimal_latitude),
            decimal_longitude = VALUES(decimal_longitude),
            coordinate_uncertainty_m = VALUES(coordinate_uncertainty_m),
            event_date = VALUES(event_date),
            basis_of_record = VALUES(basis_of_record);
        """

        sql_mysql_media = """
        INSERT INTO ocurrencia_media (gbif_id, media_url, orden)
        VALUES (%s, %s, %s)
        ON DUPLICATE KEY UPDATE
            media_url = VALUES(media_url);
        """

        cursor.executemany(sql_mysql_taxon, tuplas_taxones)
        cursor.executemany(sql_mysql_ocurrencia, tuplas_ocurrencias)
        cursor.executemany(sql_mysql_media, tuplas_media)
        conn_db.commit()

        cursor.execute(
            "SELECT especie_buscada, country_code, COUNT(*) AS filas_en_bd, SUM(total_imagenes) AS imgs FROM vw_ocurrencias_completas GROUP BY especie_buscada, country_code;"
        )
        filas_resumen = cursor.fetchall()
        df_resumen_mysql = pd.DataFrame(
            filas_resumen, columns=["especie_buscada", "country_code", "filas_en_bd", "imgs"]
        )
        print(f"[MySQL] Sincronización completada en '{config['host']}:{config['port']}/{db_name}'.")
        display(df_resumen_mysql)

    except MySQLError as err:
        print(f"[AVISO MySQL] No se pudo conectar o escribir en MySQL ({err}). Verifica el servicio MySQL 8 y tu archivo .env.")
    finally:
        if conn_db and conn_db.is_connected():
            conn_db.close()


if not df_total.empty:
    t_taxones, t_ocurrencias, t_media = preparar_registros_para_bd(df_catalogo, df_total)
    if ACTIVAR_SQLITE:
        cargar_en_sqlite(RUTA_SQLITE, t_taxones, t_ocurrencias, t_media)
    if ACTIVAR_MYSQL:
        cargar_en_mysql(MYSQL_CONFIG, t_taxones, t_ocurrencias, t_media)
else:
    print("No hay registros en df_total para insertar en base de datos.")